# Prompt engineering: the simple version

**Week 4 · NLP, Transformers & LLMs · Notebook 2**

A short, basic notebook. Every LLM call goes through one small function, `ask()`.

| Part | Topic |
|---|---|
| 1 | Connect to Azure OpenAI and say hello |
| 2 | Prompting styles: zero-shot, few-shot, role, chain of thought |
| 3 | Structured outputs (JSON) |
| 4 | Validating the JSON with Pydantic |
| 5 | Accuracy: which prompting strategy extracts best? |
| 6 | LLM as a judge |

> **Setup:** `uv add openai python-dotenv pydantic pandas jupyter`, and a `.env` file next to this notebook (see `.env.example`).
> Cells starting with **# 🤖 calls the LLM** were saved without outputs: run them yourself. Answers change between runs.

## 1. Connect to Azure OpenAI and say hello

`.env` holds the endpoint, key and deployment name, so no secret is ever typed into the notebook.

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [ ]:
import json
import os

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
api_key = os.getenv("AZURE_OPENAI_API_KEY", "")
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")

print("Endpoint set:", bool(endpoint))
print("API key set: ", bool(api_key))   # never print the key itself
print("Deployment:  ", MODEL)

client = OpenAI(base_url=f"{endpoint}/openai/v1/", api_key=api_key, timeout=60)

One helper for every call. `system` sets the model's role, `json_mode=True` forces the reply to be valid JSON.

> gpt-5.4-mini needs **`max_completion_tokens`** (it rejects the older `max_tokens`).

In [ ]:
def ask(prompt, system=None, json_mode=False, temperature=0):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    extra = {"response_format": {"type": "json_object"}} if json_mode else {}
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=500,
        **extra,
    )
    return response.choices[0].message.content

In [ ]:
# 🤖 calls the LLM
print(ask("Say hello to the Week 4 class in one short sentence."))

## 2. Prompting styles

The same review, four ways of asking.

In [ ]:
review = "The pasta was cold and the waiter ignored us, but the tiramisu was amazing."

### 2.1 Zero-shot: just ask
No examples. The model relies on what it learned in training.

In [ ]:
# 🤖 calls the LLM
print(ask(f"Is this restaurant review positive, negative or mixed?\n\nReview: {review}"))

### 2.2 Few-shot: show examples first
A few solved examples teach the format and the labels you want. Notice the answer is now a single word.

In [ ]:
# 🤖 calls the LLM
few_shot = f"""Classify each review as positive, negative or mixed. Answer with one word.

Review: Lovely food and friendly staff.
Answer: positive

Review: Rude service and the soup was salty.
Answer: negative

Review: Great burger, but we waited an hour.
Answer: mixed

Review: {review}
Answer:"""

print(ask(few_shot))

### 2.3 Role prompting: a system message
The system message sets who the model is and how it should behave for the whole conversation.

In [ ]:
# 🤖 calls the LLM
system = "You are a restaurant manager. Reply to customers politely in two sentences. Never offer refunds."
print(ask(f"Write a reply to this review: {review}", system=system))

### 2.4 Chain of thought: think step by step
For questions with several steps (maths, logic), asking the model to reason first and answer last makes mistakes less likely and easier to spot.

In [ ]:
# 🤖 calls the LLM
question = """A table of 4 orders 3 mains at £14 and 2 desserts at £6. Service is 10%.
They split the bill equally. How much does each person pay?"""

print("--- Direct ---")
print(ask(question + "\nGive only the final amount."))

print("\n--- Chain of thought ---")
print(ask(question + "\nThink step by step, then give the final line as 'Answer: £X.XX'."))

Correct answer: (3 × 14 + 2 × 6) × 1.10 / 4 = 54 × 1.10 / 4 = **£14.85**.

## 3. Structured outputs (JSON)

Free text is hard to use in code. If we ask for **JSON**, we can turn the answer straight into a Python dictionary.

`json_mode=True` sends `response_format={"type": "json_object"}`, which guarantees valid JSON (the prompt must mention JSON).
It does **not** guarantee the right keys or values: that is what Part 4 is for.

In [ ]:
# 🤖 calls the LLM
prompt = f"""Extract information from the restaurant review. Return JSON with these keys:
- "sentiment": "positive", "negative" or "mixed"
- "dish": the main dish mentioned, or null
- "mentions_service": true or false

Review: {review}"""

raw = ask(prompt, json_mode=True)
print(raw)

data = json.loads(raw)   # text -> Python dict
print(data["sentiment"], "|", data["dish"], "|", data["mentions_service"])

## 4. Validating the JSON with Pydantic

A **Pydantic** model describes the exact shape we expect. `model_validate_json` checks the reply:
right keys, right types, allowed values. If anything is wrong it raises a `ValidationError` instead of letting bad data into our app.

In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, Field, ValidationError


class ReviewInfo(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]
    dish: Optional[str] = None
    mentions_service: bool
    rating: int = Field(ge=1, le=5, description="Stars from 1 to 5")

We can test it without the LLM. A good reply passes, a bad one fails with a clear message:

In [ ]:
good = '{"sentiment": "mixed", "dish": "pasta", "mentions_service": true, "rating": 3}'
print(ReviewInfo.model_validate_json(good))

bad = '{"sentiment": "angry", "dish": "pasta", "mentions_service": "maybe", "rating": 9}'
try:
    ReviewInfo.model_validate_json(bad)
except ValidationError as error:
    print(error)

Now with the LLM. We can put the schema straight into the prompt with `model_json_schema()`, and **retry** once if validation fails, telling the model what was wrong.

In [ ]:
def extract(review, attempts=2):
    prompt = f"""Extract information from the restaurant review.
Return JSON matching this schema: {json.dumps(ReviewInfo.model_json_schema())}

Review: {review}"""
    for attempt in range(attempts):
        raw = ask(prompt, json_mode=True)
        try:
            return ReviewInfo.model_validate_json(raw)
        except ValidationError as error:
            print(f"Attempt {attempt + 1} failed validation, retrying")
            prompt += f"\n\nYour last answer was invalid: {error}. Fix it."
    return None

In [ ]:
# 🤖 calls the LLM
info = extract(review)
print(info)
print(info.model_dump())   # a normal dict, ready for an API or a database

> **Even stricter:** the API can enforce the schema itself (*structured outputs*):
> `client.chat.completions.parse(model=MODEL, messages=[...], response_format=ReviewInfo)` returns a `ReviewInfo` object directly.
> Validating with Pydantic yourself still matters, because it works with any model and any provider.

## 5. Accuracy: which prompting strategy extracts best?

To compare prompts fairly we need a small **labelled set**: reviews where we already know the right answers (the *gold* labels).
Then we run each strategy and count how many fields it gets right.

In [ ]:
labelled = pd.DataFrame([
    {"review": "The steak was perfect and the staff were lovely.",                 "sentiment": "positive", "mentions_service": True},
    {"review": "Cold fries and soggy bread. Never again.",                            "sentiment": "negative", "mentions_service": False},
    {"review": "Amazing curry, but the waiter forgot our drinks twice.",               "sentiment": "mixed",    "mentions_service": True},
    {"review": "Best pizza in town, honestly.",                                       "sentiment": "positive", "mentions_service": False},
    {"review": "The manager was rude when we complained about the burnt toast.",       "sentiment": "negative", "mentions_service": True},
    {"review": "Not bad at all, the salad was fresh and the server was quick.",         "sentiment": "positive", "mentions_service": True},
    {"review": "Lovely sushi, shame about the 40 minute wait for a table.",             "sentiment": "mixed",    "mentions_service": True},
    {"review": "I wouldn't say the burger was terrible, but I won't be back.",          "sentiment": "negative", "mentions_service": False},
    {"review": "The ramen was rich and tasty, and the price was fair.",                 "sentiment": "positive", "mentions_service": False},
    {"review": "Great cocktails, bland food.",                                          "sentiment": "mixed",    "mentions_service": False},
])
labelled

Three strategies for the same extraction. Each returns JSON with `sentiment` and `mentions_service`.

In [ ]:
TASK = """Read the restaurant review. Return JSON with:
- "sentiment": "positive", "negative" or "mixed"
- "mentions_service": true if the review talks about staff, waiters, waiting or service, else false"""

EXAMPLES = """Examples:
Review: Friendly waiter, average pasta. -> {"sentiment": "mixed", "mentions_service": true}
Review: Delicious lamb. -> {"sentiment": "positive", "mentions_service": false}
Review: We waited ages and the soup was cold. -> {"sentiment": "negative", "mentions_service": true}"""

def zero_shot(review):
    return f"{TASK}\n\nReview: {review}"

def few_shot(review):
    return f"{TASK}\n\n{EXAMPLES}\n\nReview: {review}"

def chain_of_thought(review):
    return (f"{TASK}\nAlso include a \"reasoning\" key FIRST, where you think step by step "
            f"about what is praised and what is criticised.\n\nReview: {review}")

STRATEGIES = {"zero-shot": zero_shot, "few-shot": few_shot, "chain of thought": chain_of_thought}

In [ ]:
# 🤖 calls the LLM
rows = []
for name, build_prompt in STRATEGIES.items():
    for _, row in labelled.iterrows():
        try:
            answer = json.loads(ask(build_prompt(row["review"]), json_mode=True))
        except json.JSONDecodeError:
            answer = {}   # invalid JSON counts as wrong
        rows.append({
            "strategy": name,
            "review": row["review"],
            "sentiment_ok": answer.get("sentiment") == row["sentiment"],
            "service_ok": answer.get("mentions_service") == row["mentions_service"],
        })

results = pd.DataFrame(rows)
results["both_ok"] = results["sentiment_ok"] & results["service_ok"]
results.groupby("strategy")[["sentiment_ok", "service_ok", "both_ok"]].mean().round(2)

Each number is **field-level accuracy**: the share of reviews where that field matched the gold label. `both_ok` is *exact match* (every field right).

Look at the mistakes, not just the score:

In [ ]:
# 🤖 calls the LLM
results[~results["both_ok"]]

**Things to notice**
- 10 reviews is tiny: one review = 10 percentage points. Real comparisons need 100+ labelled examples.
- Tricky reviews (negations like *"not bad"*, *"wouldn't say it was terrible"*) are where strategies differ.
- Few-shot usually helps most with the **format and the label boundaries** (what counts as *mixed*?).
- Chain of thought costs more tokens; it is worth it only if the accuracy gain is real.

## 6. LLM as a judge

Some outputs have no single right answer (a reply to a customer, a summary). Instead of a gold label we give a second LLM call a **rubric** and ask it to score the output.

### 6.1 Scoring replies with a rubric

In [ ]:
class Verdict(BaseModel):
    score: int = Field(ge=1, le=5)
    reason: str

JUDGE_SYSTEM = "You are a strict evaluator. You only score; you never rewrite the answer."

def judge_reply(review, reply):
    prompt = f"""Score the restaurant's reply to a customer review from 1 (bad) to 5 (excellent).

Rubric:
- Addresses the specific complaints in the review
- Polite and professional
- Makes no promises the restaurant cannot keep (refunds, free meals)

Review: {review}
Reply: {reply}

Return JSON: {{"score": <1-5>, "reason": "<one sentence>"}}"""
    return Verdict.model_validate_json(ask(prompt, system=JUDGE_SYSTEM, json_mode=True))

In [ ]:
# 🤖 calls the LLM
reply_a = "Thanks for visiting! We're glad you enjoyed it."
reply_b = ("We're sorry your pasta arrived cold and that you were left waiting; we've shared this with the team. "
           "Thank you for the kind words about the tiramisu.")
reply_c = "So sorry! Your next three meals are completely free, and we've fired the waiter."

for name, reply in {"A (generic)": reply_a, "B (specific)": reply_b, "C (over-promises)": reply_c}.items():
    verdict = judge_reply(review, reply)
    print(f"{name}: {verdict.score}/5  {verdict.reason}")

### 6.2 Can we trust the judge? Check it against the gold labels

A judge is only useful if it agrees with humans. We already know which extractions from Part 5 were right, so we ask the judge whether each extraction is correct and measure the **agreement**.

In [ ]:
class Check(BaseModel):
    correct: bool
    reason: str

def judge_extraction(review, sentiment, mentions_service):
    prompt = f"""Review: {review}

Someone extracted: sentiment = {sentiment}, mentions_service = {mentions_service}.
(mixed = both praise and criticism; mentions_service = staff, waiters, waiting or service.)
Is the extraction fully correct? Return JSON: {{"correct": true/false, "reason": "<one sentence>"}}"""
    return Check.model_validate_json(ask(prompt, system=JUDGE_SYSTEM, json_mode=True))

In [ ]:
# 🤖 calls the LLM
# Judge the zero-shot extractions, and also a deliberately wrong version of each (so there are mistakes to find)
checks = []
for _, row in labelled.iterrows():
    wrong_sentiment = "negative" if row["sentiment"] == "positive" else "positive"
    for sentiment, is_correct in [(row["sentiment"], True), (wrong_sentiment, False)]:
        verdict = judge_extraction(row["review"], sentiment, row["mentions_service"])
        checks.append({"review": row["review"], "given": sentiment,
                       "truly_correct": is_correct, "judge_says": verdict.correct})

checks = pd.DataFrame(checks)
agreement = (checks["truly_correct"] == checks["judge_says"]).mean()
print(f"Judge agrees with the gold labels on {agreement:.0%} of {len(checks)} checks")
checks[checks["truly_correct"] != checks["judge_says"]]

**Judge pitfalls to remember**
- **Position bias:** when comparing two answers, judges often prefer the first one. Swap the order and judge twice.
- **Verbosity bias:** longer answers tend to get higher scores. Say in the rubric that length is not quality.
- **Self-preference:** a model may favour text written by itself. Use a different model to judge when you can.
- **Calibrate first:** measure agreement with human labels (like 6.2) before trusting a judge on new data.

## Summary

| Technique | Use it when |
|---|---|
| Zero-shot | Simple tasks; a quick baseline |
| Few-shot | You need a fixed format or clear label boundaries |
| Role / system message | You need consistent behaviour, tone and rules |
| Chain of thought | Multi-step reasoning (maths, logic) |
| JSON mode | The output goes into code |
| Pydantic | Always, for structured output: check before you trust |
| Labelled set + accuracy | Choosing between prompts with evidence, not gut feeling |
| LLM as a judge | Open-ended outputs with no single right answer, after checking the judge against humans |